In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [15]:
spark = SparkSession.builder \
    .appName("Segmentacion_Perfilado_Clientes") \
    .getOrCreate()

clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)
behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)

In [16]:
behavioural_df

DataFrame[CONTRACT_ID: string, CLIENT_ID: string, DATE: date, CREDICT_CARD_BALANCE: double, CREDIT_CARD_LIMIT: double, CREDIT_CARD_DRAWINGS_ATM: double, CREDIT_CARD_DRAWINGS: double, CREDIT_CARD_DRAWINGS_POS: double, CREDIT_CARD_DRAWINGS_OTHER: double, CREDIT_CARD_PAYMENT: double, NUMBER_DRAWINGS_ATM: double, NUMBER_DRAWINGS: int, NUMBER_INSTALMENTS: double, CURRENCY: string]

In [19]:
clientes_df

DataFrame[CLIENT_ID: string, NON_COMPLIANT_CONTRACT: int, NAME_PRODUCT_TYPE: string, GENDER: string, TOTAL_INCOME: double, AMOUNT_PRODUCT: double, INSTALLMENT: double, EDUCATION: string, MARITAL_STATUS: string, HOME_SITUATION: string, REGION_SCORE: double, AGE_IN_YEARS: double, JOB_SENIORITY: double, HOME_SENIORITY: double, LAST_UPDATE: double, OWN_INSURANCE_CAR: string, CAR_AGE: double, FAMILY_SIZE: double, REACTIVE_SCORING: double, PROACTIVE_SCORING: double, BEHAVIORAL_SCORING: double, DAYS_LAST_INFO_CHANGE: double, NUMBER_OF_PRODUCTS: double, OCCUPATION: string, DIGITAL_CLIENT: int, HOME_OWNER: string, EMPLOYER_ORGANIZATION_TYPE: string, CURRENCY: string, NUM_PREVIOUS_LOAN_APP: double, LOAN_ANNUITY_PAYMENT_MAX: double, LOAN_ANNUITY_PAYMENT_MIN: double, LOAN_ANNUITY_PAYMENT_SUM: double, LOAN_APPLICATION_AMOUNT_MAX: double, LOAN_APPLICATION_AMOUNT_MIN: double, LOAN_APPLICATION_AMOUNT_SUM: double, LOAN_CREDIT_GRANTED_MAX: double, LOAN_CREDIT_GRANTED_MIN: double, LOAN_CREDIT_GRANTED_SUM

# METRICAS

In [13]:
#CLV = Customer Lifetime Value

df_clv = behavioural_df.withColumn(
    "MARGEN",
    (F.col("CREDICT_CARD_BALANCE") + F.col("CREDIT_CARD_DRAWINGS_POS")) - F.col("CREDIT_CARD_PAYMENT")
).alias("MARGEN")
df_clv = df_clv.withColumn(
    "CLV",
    F.col("MARGEN") * F.avg("CREDIT_CARD_PAYMENT")
).alias("CLV")

df_clv.select("CLIENT_ID", "MARGEN", "CLV").show(5)

AnalysisException: [MISSING_GROUP_BY] The query does not include a GROUP BY clause. Add GROUP BY or turn it into the window functions using OVER clauses.;
Aggregate [CONTRACT_ID#775, CLIENT_ID#776, DATE#777, CREDICT_CARD_BALANCE#778, CREDIT_CARD_LIMIT#779, CREDIT_CARD_DRAWINGS_ATM#780, CREDIT_CARD_DRAWINGS#781, CREDIT_CARD_DRAWINGS_POS#782, CREDIT_CARD_DRAWINGS_OTHER#783, CREDIT_CARD_PAYMENT#784, NUMBER_DRAWINGS_ATM#785, NUMBER_DRAWINGS#786, NUMBER_INSTALMENTS#787, CURRENCY#788, MARGEN#803, (MARGEN#803 * avg(CREDIT_CARD_PAYMENT#784)) AS CLV#821]
+- SubqueryAlias MARGEN
   +- Project [CONTRACT_ID#775, CLIENT_ID#776, DATE#777, CREDICT_CARD_BALANCE#778, CREDIT_CARD_LIMIT#779, CREDIT_CARD_DRAWINGS_ATM#780, CREDIT_CARD_DRAWINGS#781, CREDIT_CARD_DRAWINGS_POS#782, CREDIT_CARD_DRAWINGS_OTHER#783, CREDIT_CARD_PAYMENT#784, NUMBER_DRAWINGS_ATM#785, NUMBER_DRAWINGS#786, NUMBER_INSTALMENTS#787, CURRENCY#788, ((CREDICT_CARD_BALANCE#778 + CREDIT_CARD_DRAWINGS_POS#782) - CREDIT_CARD_PAYMENT#784) AS MA

In [17]:
# 1. (Asumiendo que df_datos es tu DataFrame con la columna MARGIN_PROXY ya calculada)
#    Y asumiendo que el DataFrame tiene una fila por CLIENTE y por MES.

# 2. Agrupar por CLIENTE y calcular las métricas necesarias:
df_clv_agg = df_clv.groupBy("CLIENT_ID").agg(
    # a) Media de Gasto Mensual (usando Retiros como proxy de gasto/revenue)
    F.avg("CREDIT_CARD_DRAWINGS").alias("AVG_MONTHLY_SPEND"),
    
    # b) Media de Margen Mensual (promedio del margen proxy calculado)
    F.avg("MARGEN").alias("AVG_MONTHLY_MARGIN_PROXY"),
    
)

# 3. Aplicar la fórmula final del CLV:
# CLV = Media Gasto Mensual * Media Margen * Antigüedad
df_clv_final = df_clv_agg.withColumn(
    "CLV",
    F.col("AVG_MONTHLY_SPEND") * F.col("AVG_MONTHLY_MARGIN_PROXY")
)

df_clv_final.show(5)

+------------+------------------+------------------------+------------------+
|   CLIENT_ID| AVG_MONTHLY_SPEND|AVG_MONTHLY_MARGIN_PROXY|               CLV|
+------------+------------------+------------------------+------------------+
|ES182405039N| 282.4540540540541|      4437.8640540540555|1253492.6934083279|
|ES182283610G|              57.6|      34.617733333333334|        1993.98144|
|ES182131592N|21.813157894736843|      219.55749999999998| 4789.242414473684|
|ES182303796D|60.472043010752685|       984.4852688172042| 59533.83551936639|
|ES182133642C|31.129411764705882|       849.7805882352943|26453.169840830455|
+------------+------------------+------------------------+------------------+
only showing top 5 rows



# Estos son los clientes Prime


In [28]:
#CAQ = Customer Acquisition Quality

df_id_clientes_behavioural = clientes_df.alias("c").join(
    behavioural_df.alias("b"),
    on="CLIENT_ID",
    how="inner"
)


# Asumimos que df_id_clientes_behavioural ya fue creado correctamente (sin el .show(5))

# 1. Calcular el Payment Ratio (Pago / Saldo Pendiente)
df_features = df_id_clientes_behavioural.withColumn(
    "PAYMENT_RATIO",
    F.col("CREDIT_CARD_PAYMENT") / F.col("CREDICT_CARD_BALANCE")
)

# 2. Manejar Nulls/Division por Cero (CORREGIDO el error lógico)
# Aseguramos que (F.col("CREDICT_CARD_BALANCE") == 0) sea una expresión BOOLEAN
df_features = df_features.withColumn(
    "PAYMENT_RATIO",
    F.when(
        (F.col("PAYMENT_RATIO").isNull()) | (F.col("CREDICT_CARD_BALANCE") == 0), 1.0
    ).otherwise(F.col("PAYMENT_RATIO"))
)

# 3. Calcular Umbrales (Q75) - Usando TOTAL_INCOME
quantile_list = [0.75]

quantiles = df_features.approxQuantile(
    ["TOTAL_INCOME", "CREDIT_CARD_LIMIT", "PAYMENT_RATIO"], 
    quantile_list,
    0.01
)

# Extraer los umbrales
income_q75 = quantiles[0][0]
limit_q75 = quantiles[1][0]
ratio_q75 = quantiles[2][0]

print(f"Umbral de Ingreso (Q75): {income_q75:,.2f}")
print(f"Umbral de Límite (Q75): {limit_q75:,.2f}")
print(f"Umbral de Ratio de Pago (Q75): {ratio_q75:,.2f}")

# 4. Filtrar y Segmentar el Perfil CAQ
df_caq_profile = df_features.filter(
    (F.col("TOTAL_INCOME") >= income_q75) &           # Ingresos Altos
    (F.col("CREDIT_CARD_LIMIT") >= limit_q75) &       # Límites Altos
    (F.col("PAYMENT_RATIO") >= 1.0)                   # Sin Atrasos (Pagan al menos el 100% de la deuda)
)

print(f"Total de clientes en el perfil CAQ: {df_caq_profile.count()}")
df_caq_profile.show(5)

print("Porcentaje de clientes en el perfil CAQ:")
total_clients = clientes_df.select("CLIENT_ID").distinct().count()
caq_clients = df_caq_profile.select("CLIENT_ID").distinct().count()
percentage_caq = (caq_clients / total_clients) * 100
print(f"{percentage_caq:.2f}% de los clientes totales están en el perfil CAQ.")


Umbral de Ingreso (Q75): 2,700.00
Umbral de Límite (Q75): 2,160.00
Umbral de Ratio de Pago (Q75): 1.00
Total de clientes en el perfil CAQ: 81440
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+------

# Estos son los clientes no Prime


In [29]:
df_features

DataFrame[CLIENT_ID: string, NON_COMPLIANT_CONTRACT: int, NAME_PRODUCT_TYPE: string, GENDER: string, TOTAL_INCOME: double, AMOUNT_PRODUCT: double, INSTALLMENT: double, EDUCATION: string, MARITAL_STATUS: string, HOME_SITUATION: string, REGION_SCORE: double, AGE_IN_YEARS: double, JOB_SENIORITY: double, HOME_SENIORITY: double, LAST_UPDATE: double, OWN_INSURANCE_CAR: string, CAR_AGE: double, FAMILY_SIZE: double, REACTIVE_SCORING: double, PROACTIVE_SCORING: double, BEHAVIORAL_SCORING: double, DAYS_LAST_INFO_CHANGE: double, NUMBER_OF_PRODUCTS: double, OCCUPATION: string, DIGITAL_CLIENT: int, HOME_OWNER: string, EMPLOYER_ORGANIZATION_TYPE: string, CURRENCY: string, NUM_PREVIOUS_LOAN_APP: double, LOAN_ANNUITY_PAYMENT_MAX: double, LOAN_ANNUITY_PAYMENT_MIN: double, LOAN_ANNUITY_PAYMENT_SUM: double, LOAN_APPLICATION_AMOUNT_MAX: double, LOAN_APPLICATION_AMOUNT_MIN: double, LOAN_APPLICATION_AMOUNT_SUM: double, LOAN_CREDIT_GRANTED_MAX: double, LOAN_CREDIT_GRANTED_MIN: double, LOAN_CREDIT_GRANTED_SUM

In [35]:
# 1. Definir los cuantiles a calcular (el 25%)
quantile_list_25 = [0.25]

# 2. Calcular los cuantiles de las métricas clave (Q25)
quantiles_25 = df_features.approxQuantile(
    ["TOTAL_INCOME", "CREDIT_CARD_LIMIT", "PAYMENT_RATIO"], 
    quantile_list_25,
    0.01 
)

# 3. Extraer los umbrales Q25
income_q25 = quantiles_25[0][0]
limit_q25 = quantiles_25[1][0]

print(f"Umbral de Ingreso (Q25): {income_q25:,.2f}")
print(f"Umbral de Límite (Q25): {limit_q25:,.2f}")

# 4. Filtrar y Segmentar el Perfil Q25 (Bajo)
df_perfil_q25 = df_features.filter(
    (F.col("TOTAL_INCOME") <= income_q25) &           # Ingresos Bajos (Bottom 25%)
    (F.col("CREDIT_CARD_LIMIT") <= limit_q25) &       # Límites Bajos (Bottom 25%)
    (F.col("PAYMENT_RATIO") < 1.0)                    # Riesgo Alto (Están revolviendo deuda)
)

# 5. Cuantificar y mostrar
num_clientes_perfil_q25 = df_perfil_q25.select("CLIENT_ID").distinct().count()

print(f"\nNúmero total de clientes únicos en el Perfil Q25 (Bajo): {num_clientes_perfil_q25}")

df_perfil_q25.show(5)

print("Porcentaje de clientes en el perfil Q25 (Bajo):")
total_clients = clientes_df.select("CLIENT_ID").distinct().count()
caq_clients = df_perfil_q25.select("CLIENT_ID").distinct().count()
percentage_caq = (caq_clients / total_clients) * 100
print(f"{percentage_caq:.2f}% de los clientes totales están en el perfil Q25 (Bajo).")

Umbral de Ingreso (Q25): 1,350.00
Umbral de Límite (Q25): 540.00

Número total de clientes únicos en el Perfil Q25 (Bajo): 1506
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+----

# Estudio demografico de lo cuartiles


In [40]:
df_caq_profile
df_perfil_q25

# Columnas demográficas (asumiendo que están todas después del join)
DEMO_COLS = ["GENDER", "MARITAL_STATUS", "AGE_IN_YEARS", "FAMILY_SIZE", "EDUCATION", "HOME_SITUATION", "JOB_SENIORITY", "HOME_SENIORITY"]

# 1. Clientes Q75 (Alto) únicos
df_q75_unique = df_caq_profile.select(*DEMO_COLS).distinct() \
    .withColumn("Segmento", F.lit("Q75_ALTO"))

# 2. Clientes Q25 (Bajo) únicos
df_q25_unique = df_perfil_q25.select(*DEMO_COLS).distinct() \
    .withColumn("Segmento", F.lit("Q25_BAJO"))

# 3. Unir los dos DataFrames
df_comparison = df_q75_unique.unionByName(df_q25_unique)

print("DataFrame de comparación creado. Total de clientes para comparar:")
df_comparison.groupBy("Segmento").count().show()

df_age_analysis = df_comparison.groupBy("Segmento").agg(
    F.avg("AGE_IN_YEARS").alias("Edad_Media"),
    F.min("AGE_IN_YEARS").alias("Edad_Min"),
    F.max("AGE_IN_YEARS").alias("Edad_Max"),
    F.stddev("AGE_IN_YEARS").alias("Desv_Estandar")
).select(
    "Segmento", 
    F.round("Edad_Media", 1).alias("Edad_Media"),
    F.col("Edad_Min"),
    F.col("Edad_Max"),
    F.round("Desv_Estandar", 1).alias("Desv_Estandar")
).orderBy("Segmento")

print("\nAnálisis Demográfico: Edad")
df_age_analysis.show()

# 1. Obtener el conteo total por segmento
total_counts = df_comparison.groupBy("Segmento").count().withColumnRenamed("count", "Total_Segmento")

# 2. Calcular la frecuencia por categoría y el porcentaje
df_gender_analysis = df_comparison.groupBy("Segmento", "GENDER").count() \
    .join(total_counts, on="Segmento") \
    .withColumn("Porcentaje", (F.col("count") / F.col("Total_Segmento")) * 100) \
    .select("Segmento", "GENDER", F.round("Porcentaje", 2).alias("Porcentaje")) \
    .orderBy("Segmento", F.desc("Porcentaje"))

print("\nAnálisis Demográfico: Distribución por Género")
df_gender_analysis.show()

# Asumimos que total_counts ya fue calculado en el paso anterior (Total_Segmento por segmento)
# Si no lo tienes, usa: total_counts = df_comparison.groupBy("Segmento").count().withColumnRenamed("count", "Total_Segmento")

# 1. Calcular la frecuencia por categoría y el porcentaje de EDUCATION
df_education_analysis = df_comparison.groupBy("Segmento", "EDUCATION").count() \
    .join(total_counts, on="Segmento") \
    .withColumn("Porcentaje", (F.col("count") / F.col("Total_Segmento")) * 100) \
    .select("Segmento", "EDUCATION", F.round("Porcentaje", 2).alias("Porcentaje")) \
    .orderBy("Segmento", F.desc("Porcentaje"))

print("\nAnálisis Demográfico: Nivel de Estudios (EDUCATION)")
df_education_analysis.show(truncate=False)

df_family_size_analysis = df_comparison.groupBy("Segmento").agg(
    F.avg("FAMILY_SIZE").alias("FamilySize_Avg"),
    F.min("FAMILY_SIZE").alias("FamilySize_Min"),
    F.max("FAMILY_SIZE").alias("FamilySize_Max"),
    F.stddev("FAMILY_SIZE").alias("FamilySize_StdDev")
).select(
    "Segmento", 
    F.round("FamilySize_Avg", 2).alias("Tamaño_Familia_Media"),
    F.col("FamilySize_Min").alias("Tamaño_Familia_Min"),
    F.col("FamilySize_Max").alias("Tamaño_Familia_Max"),
    F.round("FamilySize_StdDev", 2).alias("Desv_Estandar")
).orderBy("Segmento")

print("\nAnálisis Demográfico: Tamaño de la Familia (FAMILY_SIZE)")
df_family_size_analysis.show()

DataFrame de comparación creado. Total de clientes para comparar:
+--------+-----+
|Segmento|count|
+--------+-----+
|Q75_ALTO| 6415|
|Q25_BAJO| 1505|
+--------+-----+


Análisis Demográfico: Edad
+--------+----------+------------------+-----------------+-------------+
|Segmento|Edad_Media|          Edad_Min|         Edad_Max|Desv_Estandar|
+--------+----------+------------------+-----------------+-------------+
|Q25_BAJO|      42.9|22.019178082191782|66.36986301369863|         11.2|
|Q75_ALTO|      44.0| 22.18082191780822|65.67123287671232|          9.4|
+--------+----------+------------------+-----------------+-------------+


Análisis Demográfico: Distribución por Género
+--------+------+----------+
|Segmento|GENDER|Porcentaje|
+--------+------+----------+
|Q25_BAJO|     F|     80.07|
|Q25_BAJO|     M|     19.93|
|Q75_ALTO|     F|     56.49|
|Q75_ALTO|     M|     43.51|
+--------+------+----------+


Análisis Demográfico: Nivel de Estudios (EDUCATION)
+--------+---------------------